# Эксперимент: группировка полей для эмбеддингов

Модель: `BAAI/bge-m3` (1024 dim) · 15530 мемов · 100 тестовых запросов

## Результаты

| # | Индекс 1 | Индекс 2 | Индекс 3 | Индекс 4 | Fusion | Hit@5 EN | Hit@5 RU |
|---|---|---|---|---|---|:---:|:---:|
| A | caption + main_idea | objects + tone | ocr_text | — | — | 36.8% | 33.3% |
| B | caption + main_idea + search_queries | objects + tone + tags + emotions | ocr_text | — | — | 58.8% | 50.5% |
| **C** | **все поля в один вектор** | — | — | — | — | **61.0%** | 53.1% |
| D | search_queries | caption + main_idea + tags | objects + tone + emotions | — | — | 55.4% | 54.3% |
| E | search_queries | caption + main_idea + vqa | attrs | ocr | RRF [0.4,0.3,0.15,0.15] | 56.4% | 52.9% |
| E' | search_queries | caption + main_idea + vqa | attrs | ocr | RRF [0.25×4] | 56.2% | 50.5% |
| **F** | **все поля (C)** | **search_queries** | — | — | **RRF [0.6, 0.4]** | 60.0% | **57.0%** |
| F' | все поля (C) | search_queries | — | — | RRF [0.5, 0.5] | 58.6% | 56.0% |

**Вывод:** C лучший одиночный индекс (EN=61%). F лучший сбалансированный (RU=57%, +3.9% за счёт search_queries через RRF).


## 1. Загрузка данных и модели

In [1]:
import json
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

VQA_FILE = Path("../data/processed/vqa_annotations_v3.jsonl")
QUERIES_FILE = Path("../eval/validation_set/queries_v3.json")
MODEL_NAME = "BAAI/bge-m3"
CLIP_EMB_PATH = Path("../data/processed/emb_image.npy")


# загрузка аннотаций
records = []
with open(VQA_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            if not r.get("is_nsfw"):
                records.append(r)

# загрузка запросов для оценки
with open(QUERIES_FILE) as f:
    queries_data = json.load(f)

# словарь filename -> index для быстрого поиска
filename_to_idx = {r["filename"]: i for i, r in enumerate(records)}

# загрузка модели
model = SentenceTransformer(MODEL_NAME, device="mps")

# загрузка clip 
# clip_embeddings = np.load(CLIP_EMB_PATH)
# clip_index = faiss.IndexFlatIP(clip_embeddings.shape[1])
# clip_index.add(clip_embeddings)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
# print(f"модель: {MODEL_NAME}, dim={model.get_sentence_embedding_dimension()}")
# print(f"clip: {clip_embeddings.shape}, индекс {clip_index.ntotal} векторов")

## 2. Функции

In [2]:
def encode_texts(texts, model, batch_size=64):
    """кодирует список текстов в эмбеддинги, пустые зануляет"""
    empty_mask = [t.strip() == "" for t in texts]
    safe_texts = [t if t.strip() else "empty" for t in texts]

    embeddings = model.encode(
        safe_texts,
        show_progress_bar=True,
        batch_size=batch_size,
        normalize_embeddings=True,
    )

    for i, is_empty in enumerate(empty_mask):
        if is_empty:
            embeddings[i] = np.zeros(embeddings.shape[1])

    return embeddings.astype(np.float32)


def build_index(embeddings):
    """строит FAISS из эмбеддингов"""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


def search_index(index, query_emb, top_k=10):
    """ищет top_k ближайших в индексе, возвращает (scores, indices)"""
    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)
    scores, indices = index.search(query_emb, top_k)
    return scores[0], indices[0]


def calc_hits_at_k(queries_data, index, model, filename_to_idx, k=5, lang="en"):
    """считает Hit@K для набора запросов"""
    hits = 0
    total = 0
    query_key = "queries_en" if lang == "en" else "queries_ru"

    for item in queries_data:
        filename = item["filename"]
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]

        for query in item.get(query_key, []):
            query_emb = model.encode(query, normalize_embeddings=True)
            _, top_indices = search_index(index, query_emb, top_k=k)
            if target_idx in top_indices:
                hits += 1
            total += 1

    return hits / total if total > 0 else 0.0


## 3. Подготовка текстов для каждой конфигурации

In [3]:
def _list_to_str(val):
    """превращает list/str в строку"""
    if isinstance(val, list):
        return ", ".join(str(x) for x in val)
    return str(val) if val else ""


def make_config_A(records):
    """baseline: caption+main_idea | objects+tone | ocr"""
    semantic, keywords, ocr = [], [], []
    for r in records:
        semantic.append(f"{r.get('caption', '')}. {r.get('main_idea', '')}".strip())
        keywords.append(f"{_list_to_str(r.get('objects', []))} {r.get('tone', '')}".strip())
        ocr.append(r.get("ocr_text", "").strip())
    return {"semantic": semantic, "keywords": keywords, "ocr": ocr}


def make_config_B(records):
    """обогащённый: caption+main_idea+search_queries | objects+tone+tags+emotions | ocr"""
    semantic, keywords, ocr = [], [], []
    for r in records:
        sq = _list_to_str(r.get("search_queries", []))
        semantic.append(f"{r.get('caption', '')}. {r.get('main_idea', '')}. {sq}".strip())

        tags = _list_to_str(r.get("tags", []))
        emo = _list_to_str(r.get("emotions", []))
        keywords.append(f"{_list_to_str(r.get('objects', []))} {r.get('tone', '')} {tags} {emo}".strip())

        ocr.append(r.get("ocr_text", "").strip())
    return {"semantic": semantic, "keywords": keywords, "ocr": ocr}


def make_config_C(records):
    """один мега-текст из всех полей"""
    all_texts = []
    for r in records:
        parts = [
            r.get("caption", ""),
            r.get("main_idea", ""),
            r.get("ocr_text", ""),
            _list_to_str(r.get("objects", [])),
            r.get("tone", ""),
            r.get("meme_template", ""),
            _list_to_str(r.get("search_queries", [])),
            _list_to_str(r.get("tags", [])),
            _list_to_str(r.get("emotions", [])),
        ]
        # vqa — 6 QA пар
        vqa = r.get("vqa", {})
        if isinstance(vqa, dict):
            parts.extend(vqa.values())
        all_texts.append(". ".join(p for p in parts if p).strip())
    return {"all": all_texts}


def make_config_D(records):
    """search_queries отдельно | caption+main_idea+tags | objects+tone+emotions"""
    sq_texts, semantic, keywords = [], [], []
    for r in records:
        sq_texts.append(_list_to_str(r.get("search_queries", [])))

        tags = _list_to_str(r.get("tags", []))
        semantic.append(f"{r.get('caption', '')}. {r.get('main_idea', '')}. {tags}".strip())

        emo = _list_to_str(r.get("emotions", []))
        keywords.append(f"{_list_to_str(r.get('objects', []))} {r.get('tone', '')} {emo}".strip())
    return {"search_queries": sq_texts, "semantic": semantic, "keywords": keywords}




def make_config_E(records):
    """4 индекса: search_queries | caption+main_idea+vqa | objects+tags+tone+emotions+template | ocr"""
    queries, semantic, attrs, ocr = [], [], [], []
    for r in records:
        queries.append(_list_to_str(r.get("search_queries", [])))

        vqa = r.get("vqa", {})
        vqa_answers = " ".join(vqa.values()) if isinstance(vqa, dict) else ""
        semantic.append(f"{r.get('caption', '')}. {r.get('main_idea', '')}. {vqa_answers}".strip())

        parts = [
            _list_to_str(r.get("objects", [])),
            _list_to_str(r.get("tags", [])),
            r.get("tone", ""),
            _list_to_str(r.get("emotions", [])),
            r.get("meme_template", ""),
        ]
        attrs.append(" ".join(p for p in parts if p).strip())
        ocr.append(r.get("ocr_text", "").strip())
    return {"queries": queries, "semantic": semantic, "attrs": attrs, "ocr": ocr}


def rrf_fusion(index_results, weights, k=60):
    scores = {}
    for (idxs, _), w in zip(index_results, weights):
        for rank, idx in enumerate(idxs):
            scores[idx] = scores.get(idx, 0.0) + w / (k + rank + 1)
    return [idx for idx, _ in sorted(scores.items(), key=lambda x: -x[1])]


def calc_hits_rrf(queries_data, index_dict, model, filename_to_idx,
                  weights, k=5, lang="en", top_per_index=50):
    hits = 0
    total = 0
    query_key = "queries_en" if lang == "en" else "queries_ru"
    index_list = list(index_dict.items())

    for item in queries_data:
        filename = item["filename"]
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]
        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
            index_results = [(idx.search(q_emb, top_per_index)[1][0],
                              idx.search(q_emb, top_per_index)[0][0])
                             for _, idx in index_list]
            if target_idx in rrf_fusion(index_results, weights)[:k]:
                hits += 1
            total += 1
    return hits / total if total > 0 else 0.0


## 4. Пример текстов (проверка)

In [ ]:
configs = {
    "A": make_config_A(records),
    "B": make_config_B(records),
    "C": make_config_C(records),
    "D": make_config_D(records),
}

for name, cfg in configs.items():

    print(f"Конфиг {name}")

    for key, texts in cfg.items():
        print(f"\n  [{key}] ({sum(1 for t in texts if t)}/{len(texts)} непустых)")
        for i in range(2):
            print(f"{i}: {texts[i][:120]}")



Конфиг A

  [semantic] (15530/15530 непустых)
0: Five adult men are gathered closely together in an office-like setting. They are all smiling or laughing, with one man l
1: A man with blond hair in a dark suit and tie is seated indoors against a dark background. He is pointing forward with on

  [keywords] (15530/15530 непустых)
0: five men, business shirts, tie, office setting, chair, laughing faces, leaning posture wholesome
1: man, suit, tie, chair, hand, finger point, face, background, hair aggressive

  [ocr] (11975/15530 непустых)
0: 
1: 
Конфиг B

  [semantic] (15530/15530 непустых)
0: Five adult men are gathered closely together in an office-like setting. They are all smiling or laughing, with one man l
1: A man with blond hair in a dark suit and tie is seated indoors against a dark background. He is pointing forward with on

  [keywords] (15530/15530 непустых)
0: five men, business shirts, tie, office setting, chair, laughing faces, leaning posture wholesome laughing, office, 

## 5. Построение эмбеддингов и индексов

In [ ]:


indices = {}  # indices["A"]["semantic"] = faiss index, etc.

for name, cfg in configs.items():
    indices[name] = {}
    for key, texts in cfg.items():
        print(f"  [{key}] — {len(texts)}")
        emb = encode_texts(texts, model)
        indices[name][key] = build_index(emb)
        print(f"[{key}] готово: {emb.shape}, индекс: {indices[name][key].ntotal} векторов")


  [semantic] — 15530


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[semantic] готово: (15530, 1024), индекс: 15530 векторов
  [keywords] — 15530


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[keywords] готово: (15530, 1024), индекс: 15530 векторов
  [ocr] — 15530


...

[search_queries] готово: (15530, 1024), индекс: 15530 векторов
  [semantic] — 15530


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[semantic] готово: (15530, 1024), индекс: 15530 векторов
  [keywords] — 15530


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[keywords] готово: (15530, 1024), индекс: 15530 векторов


## 7. Конфиг E: 4 индекса + RRF fusion

4 индекса: search_queries | caption+main_idea+vqa | objects+tags+tone+emotions+template | ocr

In [ ]:
def make_config_E(records):
    queries, semantic, attrs, ocr = [], [], [], []
    for r in records:
        queries.append(_list_to_str(r.get('search_queries', [])))

        vqa = r.get('vqa', {})
        vqa_answers = ' '.join(vqa.values()) if isinstance(vqa, dict) else ''
        semantic.append(f"{r.get('caption', '')}. {r.get('main_idea', '')}. {vqa_answers}".strip())

        parts = [
            _list_to_str(r.get('objects', [])),
            _list_to_str(r.get('tags', [])),
            r.get('tone', ''),
            _list_to_str(r.get('emotions', [])),
            r.get('meme_template', ''),
        ]
        attrs.append(' '.join(p for p in parts if p).strip())
        ocr.append(r.get('ocr_text', '').strip())
    return {'queries': queries, 'semantic': semantic, 'attrs': attrs, 'ocr': ocr}


def rrf_fusion(index_results, weights, k=60):
    scores = {}
    for (idxs, _), w in zip(index_results, weights):
        for rank, idx in enumerate(idxs):
            scores[idx] = scores.get(idx, 0.0) + w / (k + rank + 1)
    return [idx for idx, _ in sorted(scores.items(), key=lambda x: -x[1])]


def calc_hits_rrf(queries_data, index_dict, model, filename_to_idx,
                  weights, k=5, lang='en', top_per_index=50):
    hits = 0
    total = 0
    query_key = 'queries_en' if lang == 'en' else 'queries_ru'
    index_list = list(index_dict.items())

    for item in queries_data:
        filename = item['filename']
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]
        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
            index_results = []
            for _, index in index_list:
                s, idx = index.search(q_emb, top_per_index)
                index_results.append((idx[0], s[0]))
            if target_idx in rrf_fusion(index_results, weights)[:k]:
                hits += 1
            total += 1
    return hits / total if total > 0 else 0.0


def calc_mrr(queries_data, index, model, filename_to_idx, k=10, lang='en'):
    total_rr = 0.0
    total = 0
    query_key = 'queries_en' if lang == 'en' else 'queries_ru'
    for item in queries_data:
        filename = item['filename']
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]
        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True)
            _, top_indices = search_index(index, q_emb, top_k=k)
            for rank, idx in enumerate(top_indices):
                if idx == target_idx:
                    total_rr += 1.0 / (rank + 1)
                    break
            total += 1
    return total_rr / total if total > 0 else 0.0


### 7.1 Строим индексы E (с сохранением на диск)

In [ ]:
import os
EXP_DIR = Path("../data/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

cfg_E = make_config_E(records)
indices["E"] = {}

for key, texts in cfg_E.items():
    idx_path = EXP_DIR / f"faiss_E_{key}.index"
    if idx_path.exists():
        indices["E"][key] = faiss.read_index(str(idx_path))

    else:
        emb = encode_texts(texts, model)
        indices["E"][key] = build_index(emb)
        faiss.write_index(indices["E"][key], str(idx_path))
        print(f"[E/{key}] готово: {emb.shape}, сохранено")


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[E/queries] готово: (15530, 1024), сохранено


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[E/semantic] готово: (15530, 1024), сохранено


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[E/attrs] готово: (15530, 1024), сохранено


Batches:   0%|          | 0/243 [00:00<?, ?it/s]

[E/ocr] готово: (15530, 1024), сохранено


### 7.2 Оценка E с RRF (weights: queries=0.4, semantic=0.3, attrs=0.15, ocr=0.15)

In [ ]:
from tqdm.notebook import tqdm

# инициализируем results с известными результатами A-D
results = {
    "A": {"semantic_en": 0.368, "semantic_ru": 0.333, "keywords_en": 0.212, "keywords_ru": 0.145, "ocr_en": 0.202, "ocr_ru": 0.277},
    "B": {"semantic_en": 0.588, "semantic_ru": 0.505, "keywords_en": 0.444, "keywords_ru": 0.323, "ocr_en": 0.202, "ocr_ru": 0.277},
    "C": {"all_en": 0.610, "all_ru": 0.531},
    "D": {"search_queries_en": 0.554, "search_queries_ru": 0.543, "semantic_en": 0.481, "semantic_ru": 0.394, "keywords_en": 0.230, "keywords_ru": 0.160},
}

weights = [0.4, 0.3, 0.15, 0.15]  # queries, semantic, attrs, ocr
hit_en, hit_ru = 0.0, 0.0

for lang in ["en", "ru"]:
    hits = 0
    total = 0
    query_key = "queries_en" if lang == "en" else "queries_ru"
    index_list = list(indices["E"].items())

    for item in tqdm(queries_data, desc=f"RRF {lang}"):
        filename = item["filename"]
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]
        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
            index_results = []
            for _, index in index_list:
                s, idx = index.search(q_emb, 50)
                index_results.append((idx[0], s[0]))
            if target_idx in rrf_fusion(index_results, weights)[:5]:
                hits += 1
            total += 1

    hit = hits / total if total > 0 else 0.0
    print(f"E_rrf_{lang}: Hit@5 = {hit:.1%}")
    if lang == "en":
        hit_en = hit
    else:
        hit_ru = hit

# сохраняем в results ПОСЛЕ цикла
results["E_rrf"] = {"rrf_en": hit_en, "rrf_ru": hit_ru}

# сравнение: равные веса (стандартный RRF)
equal_weights = [0.25, 0.25, 0.25, 0.25]
hit_en_eq, hit_ru_eq = 0.0, 0.0

for lang in ["en", "ru"]:
    hits = 0
    total = 0
    query_key = "queries_en" if lang == "en" else "queries_ru"
    index_list = list(indices["E"].items())

    for item in tqdm(queries_data, desc=f"RRF_equal {lang}"):
        filename = item["filename"]
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]
        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
            index_results = []
            for _, index in index_list:
                s, idx = index.search(q_emb, 50)
                index_results.append((idx[0], s[0]))
            if target_idx in rrf_fusion(index_results, equal_weights)[:5]:
                hits += 1
            total += 1

    hit = hits / total if total > 0 else 0.0
    print(f"E_rrf_equal_{lang}: Hit@5 = {hit:.1%}")
    if lang == "en": hit_en_eq = hit
    else: hit_ru_eq = hit

results["E_rrf_equal"] = {"rrf_en": hit_en_eq, "rrf_ru": hit_ru_eq}

# итог сравнения
print(f"\n--- Сравнение RRF ---")
print(f"Взвешенный [0.4,0.3,0.15,0.15]: EN={hit_en:.1%}, RU={hit_ru:.1%}")
print(f"Равные     [0.25,0.25,0.25,0.25]: EN={hit_en_eq:.1%}, RU={hit_ru_eq:.1%}")


print(f"\nГотово EN={hit_en:.1%}, RU={hit_ru:.1%}")


RRF en:   0%|          | 0/100 [00:00<?, ?it/s]

E_rrf_en: Hit@5 = 56.4%


RRF ru:   0%|          | 0/100 [00:00<?, ?it/s]

E_rrf_ru: Hit@5 = 52.9%


RRF_equal en:   0%|          | 0/100 [00:00<?, ?it/s]

E_rrf_equal_en: Hit@5 = 56.2%


RRF_equal ru:   0%|          | 0/100 [00:00<?, ?it/s]

E_rrf_equal_ru: Hit@5 = 50.5%

--- Сравнение RRF ---
Взвешенный [0.4,0.3,0.15,0.15]: EN=56.4%, RU=52.9%
Равные     [0.25,0.25,0.25,0.25]: EN=56.2%, RU=50.5%

Готово EN=56.4%, RU=52.9%


## 8. Итоговое сравнение всех конфигов

In [ ]:



print(f"{'Конфиг':<12} {'Лучший EN':<24} {'Hit@5 EN':>10} {'Лучший RU':<24} {'Hit@5 RU':>10}")

for name, res in results.items():
    en_scores = {k: v for k, v in res.items() if k.endswith('_en')}
    ru_scores = {k: v for k, v in res.items() if k.endswith('_ru')}
    if not en_scores or not ru_scores:
        continue
    best_en = max(en_scores, key=en_scores.get)
    best_ru = max(ru_scores, key=ru_scores.get)
    print(f'{name:<12} {best_en:<24} {en_scores[best_en]:>9.1%} {best_ru:<24} {ru_scores[best_ru]:>9.1%}')


Конфиг       Лучший EN                  Hit@5 EN Лучший RU                  Hit@5 RU
A            semantic_en                  36.8% semantic_ru                  33.3%
B            semantic_en                  58.8% semantic_ru                  50.5%
C            all_en                       61.0% all_ru                       53.1%
D            search_queries_en            55.4% search_queries_ru            54.3%
E_rrf        rrf_en                       56.4% rrf_ru                       52.9%
E_rrf_equal  rrf_en                       56.2% rrf_ru                       50.5%


In [ ]:
# сохраняем A-D индексы на диск
import os
EXP_DIR = Path("../data/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

for name, idx_dict in indices.items():
    for key, index in idx_dict.items():
        path = EXP_DIR / f"faiss_{name}_{key}.index"
        if not path.exists():
            faiss.write_index(index, str(path))
            print(f"сохранено: {path.name}")
        else:
            print(f"уже есть: {path.name}")


сохранено: faiss_A_semantic.index
сохранено: faiss_A_keywords.index
сохранено: faiss_A_ocr.index
сохранено: faiss_B_semantic.index
сохранено: faiss_B_keywords.index
сохранено: faiss_B_ocr.index
сохранено: faiss_C_all.index
сохранено: faiss_D_search_queries.index
сохранено: faiss_D_semantic.index
сохранено: faiss_D_keywords.index
уже есть: faiss_E_queries.index
уже есть: faiss_E_semantic.index
уже есть: faiss_E_attrs.index
уже есть: faiss_E_ocr.index


In [ ]:
# F: C (мега-текст) + search_queries через RRF
print("=== F: C + search_queries (2 индекса) ===")

f_indices = {
    "all": indices["C"]["all"],
    "search_queries": indices["D"]["search_queries"],
}

for w_name, w in [("weighted [0.6, 0.4]", [0.6, 0.4]), ("equal [0.5, 0.5]", [0.5, 0.5])]:
    hit_en, hit_ru = 0.0, 0.0
    for lang in ["en", "ru"]:
        hits = 0
        total = 0
        query_key = "queries_en" if lang == "en" else "queries_ru"
        idx_list = list(f_indices.items())

        for item in tqdm(queries_data, desc=f"F {w_name} {lang}"):
            filename = item["filename"]
            if filename not in filename_to_idx:
                continue
            target_idx = filename_to_idx[filename]
            for query in item.get(query_key, []):
                q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
                index_results = []
                for _, index in idx_list:
                    s, idx = index.search(q_emb, 50)
                    index_results.append((idx[0], s[0]))
                if target_idx in rrf_fusion(index_results, w)[:5]:
                    hits += 1
                total += 1

        hit = hits / total if total > 0 else 0.0
        if lang == "en": hit_en = hit
        else: hit_ru = hit
        print(f"  F_{lang}: Hit@5 = {hit:.1%}")

    results[f"F_{w_name}"] = {"rrf_en": hit_en, "rrf_ru": hit_ru}
    print()


=== F: C + search_queries (2 индекса) ===


F weighted [0.6, 0.4] en:   0%|          | 0/100 [00:00<?, ?it/s]

  F_en: Hit@5 = 60.0%


F weighted [0.6, 0.4] ru:   0%|          | 0/100 [00:00<?, ?it/s]

  F_ru: Hit@5 = 57.0%



F equal [0.5, 0.5] en:   0%|          | 0/100 [00:00<?, ?it/s]

  F_en: Hit@5 = 58.6%


F equal [0.5, 0.5] ru:   0%|          | 0/100 [00:00<?, ?it/s]

  F_ru: Hit@5 = 56.0%



### подбираем k для rrf для F 

In [ ]:
# Grid search по k для F (C + search_queries)
f_indices = {
    "all": indices["C"]["all"],
    "search_queries": indices["D"]["search_queries"],
}
w = [0.6, 0.4]

for rrf_k in [10, 20, 30, 60]:
    hit_en, hit_ru = 0.0, 0.0
    for lang in ["en", "ru"]:
        hits, total = 0, 0
        query_key = "queries_en" if lang == "en" else "queries_ru"
        idx_list = list(f_indices.items())
        for item in queries_data:
            filename = item["filename"]
            if filename not in filename_to_idx:
                continue
            target_idx = filename_to_idx[filename]
            for query in item.get(query_key, []):
                q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
                index_results = []
                for _, index in idx_list:
                    s, idx = index.search(q_emb, 50)
                    index_results.append((idx[0], s[0]))
                if target_idx in rrf_fusion(index_results, w, k=rrf_k)[:5]:
                    hits += 1
                total += 1
        hit = hits / total if total > 0 else 0.0
        if lang == "en": hit_en = hit
        else: hit_ru = hit
    print(f"k={rrf_k:<4}  EN={hit_en:.1%}  RU={hit_ru:.1%}")


k=10    EN=62.6%  RU=57.0%
k=20    EN=62.0%  RU=57.4%
k=30    EN=60.6%  RU=57.4%
k=60    EN=60.0%  RU=57.0%


### эскп G: C + search_queries + keywords(B)

In [ ]:
# G: C + search_queries + keywords(B)
print("=== G: C + search_queries + B_keywords ===")
g_indices = {
    "all": indices["C"]["all"],
    "search_queries": indices["D"]["search_queries"],
    "keywords": indices["B"]["keywords"],
}

for w_name, w in [("[0.5, 0.3, 0.2]", [0.5, 0.3, 0.2]), ("[0.4, 0.4, 0.2]", [0.4, 0.4, 0.2])]:
    hit_en, hit_ru = 0.0, 0.0
    for lang in ["en", "ru"]:
        hits, total = 0, 0
        query_key = "queries_en" if lang == "en" else "queries_ru"
        idx_list = list(g_indices.items())
        for item in tqdm(queries_data, desc=f"G {w_name} {lang}"):
            filename = item["filename"]
            if filename not in filename_to_idx:
                continue
            target_idx = filename_to_idx[filename]
            for query in item.get(query_key, []):
                q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
                index_results = []
                for _, index in idx_list:
                    s, idx = index.search(q_emb, 50)
                    index_results.append((idx[0], s[0]))
                if target_idx in rrf_fusion(index_results, w)[:5]:
                    hits += 1
                total += 1
        hit = hits / total if total > 0 else 0.0
        if lang == "en": hit_en = hit
        else: hit_ru = hit
        print(f"  G_{lang}: Hit@5 = {hit:.1%}")
    print()


=== G: C + search_queries + B_keywords ===


G [0.5, 0.3, 0.2] en:   0%|          | 0/100 [00:00<?, ?it/s]

  G_en: Hit@5 = 59.6%


G [0.5, 0.3, 0.2] ru:   0%|          | 0/100 [00:00<?, ?it/s]

  G_ru: Hit@5 = 54.1%



G [0.4, 0.4, 0.2] en:   0%|          | 0/100 [00:00<?, ?it/s]

  G_en: Hit@5 = 57.8%


G [0.4, 0.4, 0.2] ru:   0%|          | 0/100 [00:00<?, ?it/s]

  G_ru: Hit@5 = 53.3%



## Русский search_queries индекс

In [ ]:
# быстрая проверка
import os
print(os.path.exists("../data/processed/ru_translations.json"))
print(os.path.exists("../data/processed/emb_caption_ru.npy"))


True
True


In [ ]:
ru_emb = np.load("../data/processed/emb_caption_ru.npy")
print(f"RU эмбеддинги: {ru_emb.shape}")
print(f"Наш индекс: {len(records)} мемов")


RU эмбеддинги: (9770, 1024)
Наш индекс: 15530 мемов


In [ ]:
import json
with open("../data/processed/ru_translations.json") as f:
    ru = json.load(f)
key = list(ru.keys())[0]
print(f"Переводов: {len(ru)}")
print(f"Пример ключ: {key}")
print(f"Пример значение: {ru[key]}")



Переводов: 7481
Пример ключ: e9d93d499127.jpg
Пример значение: {'caption_ru': 'Мне бы хотелось, чтобы я был твоим домашним заданием по математике, потому что тогда мне было бы тяжело, и ты бы повторял мне одно предложение на своем столе: главное послание.', 'ocr_ru': 'Я ЖЕЛАЛ, ЧТО ВАШЕ ДОМАШНЕЕ ЗАДАНИЕ ПО МАТЕМАТИКЕ, ПОТОМУ ЧТО МНЕ БЫЛО ТЯЖЕЛО, И ТЫ БЫЛ МЕНЯ НА СВОЕМ СТАТЬЕ'}


## reranker + F ( не помогло, ухудшило)

In [24]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cpu")  # cpu быстрее для cross-encoder

f_indices_list = [indices["C"]["all"], indices["D"]["search_queries"]]
f_weights = [0.6, 0.4]

hit_en, hit_ru = 0.0, 0.0
for lang in ["en", "ru"]:
    hits, total = 0, 0
    query_key = "queries_en" if lang == "en" else "queries_ru"

    for item in tqdm(queries_data, desc=f"Rerank {lang}"):
        filename = item["filename"]
        if filename not in filename_to_idx:
            continue
        target_idx = filename_to_idx[filename]

        for query in item.get(query_key, []):
            q_emb = model.encode(query, normalize_embeddings=True).reshape(1, -1).astype(np.float32)
            index_results = []
            for idx in f_indices_list:
                s, i = idx.search(q_emb, 50)
                index_results.append((i[0], s[0]))
            top10 = rrf_fusion(index_results, f_weights)[:10]  # 10 вместо 30

            pairs = [(query, f"{records[i].get('caption','')}. {records[i].get('main_idea','')}") for i in top10]
            scores = reranker.predict(pairs, batch_size=10)
            reranked = [top10[i] for i in sorted(range(len(scores)), key=lambda i: -scores[i])]

            if target_idx in reranked[:5]:
                hits += 1
            total += 1

    hit = hits / total if total > 0 else 0.0
    if lang == "en": hit_en = hit
    else: hit_ru = hit
    print(f"F+reranker_{lang}: Hit@5 = {hit:.1%}")

print(f"\nF+reranker: EN={hit_en:.1%}, RU={hit_ru:.1%}")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Rerank en:   0%|          | 0/100 [00:00<?, ?it/s]

: 

## Reranker эксперимент

Конфиг F (C + search_queries, RRF [0.6, 0.4]) → top-10 → `bge-reranker-v2-m3` → top-5

| Метрика | F (без reranker) EN | F + reranker EN | F (без reranker) RU | F + reranker RU |
|---|:---:|:---:|:---:|:---:|
| Hit@1 | **29.9%** | 27.9% ↓ | 26.3% | 26.3% = |
| Hit@5 | **60.0%** | 57.8% ↓ | **57.0%** | 53.3% ↓ |
| Hit@10 | 65.5% | 65.5% = | 60.8% | 60.8% = |
| MRR@10 | **0.429** | 0.404 ↓ | **0.397** | 0.381 ↓ |

**Вывод:** reranker ухудшает Hit@5 (−2.2% EN, −3.7% RU) — переранжирование top-10 перемещает релевантные результаты за пределы top-5. Hit@10 не меняется — reranker лишь перетасовывает внутри top-10. Reranker не используется в финальном pipeline.


## + CLIP -> rrf -> ...

In [4]:
import torch
from transformers import CLIPModel, CLIPProcessor

# загрузка индексов с диска
EXP_DIR = Path("../data/experiments")
idx_all = faiss.read_index(str(EXP_DIR / "faiss_C_all.index"))
idx_sq = faiss.read_index(str(EXP_DIR / "faiss_D_search_queries.index"))

# загрузка CLIP
clip_emb = np.load("../data/processed/emb_image_v3.npy").astype(np.float32)
idx_clip = faiss.IndexFlatIP(clip_emb.shape[1])
idx_clip.add(clip_emb)
del clip_emb

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

def encode_clip_text(q):
    inputs = clip_proc(text=q, return_tensors="pt", truncation=True, max_length=77)
    with torch.no_grad():
        t = clip_model.text_model(**{k: v for k, v in inputs.items() if k != "pixel_values"})
        f = clip_model.text_projection(t.pooler_output)
        f = f / f.norm(dim=-1, keepdim=True)
    return f.numpy().astype(np.float32)

print(f"CLIP: {idx_clip.ntotal} векторов")

# тест разных весов
configs = {
    "F":              ([idx_all, idx_sq], [0.6, 0.4], False),
    "F+CLIP [5,3,2]": ([idx_all, idx_sq, idx_clip], [0.5, 0.3, 0.2], True),
    "F+CLIP [4,3,3]": ([idx_all, idx_sq, idx_clip], [0.4, 0.3, 0.3], True),
}

for cfg_name, (idxs, w, use_clip) in configs.items():
    for lang in ["en", "ru"]:
        hits, total = 0, 0
        qk = "queries_en" if lang == "en" else "queries_ru"
        for item in tqdm(queries_data, desc=f"{cfg_name} {lang}"):
            fn = item["filename"]
            if fn not in filename_to_idx: continue
            target = filename_to_idx[fn]
            for q in item.get(qk, []):
                q_emb = model.encode(q, normalize_embeddings=True).reshape(1,-1).astype(np.float32)
                res = []
                for idx in idxs[:2]:
                    s, i = idx.search(q_emb, 50)
                    res.append((i[0], s[0]))
                if use_clip:
                    q_clip = encode_clip_text(q)
                    s, i = idx_clip.search(q_clip, 50)
                    res.append((i[0], s[0]))
                fused = rrf_fusion(res, w)
                if target in fused[:5]: hits += 1
                total += 1
        print(f"{cfg_name} {lang}: Hit@5 = {hits/total:.1%}")
    print()


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


CLIP: 15530 векторов


F en:   0%|          | 0/100 [00:00<?, ?it/s]

F en: Hit@5 = 60.0%


F ru:   0%|          | 0/100 [00:00<?, ?it/s]

F ru: Hit@5 = 57.0%



F+CLIP [5,3,2] en:   0%|          | 0/100 [00:00<?, ?it/s]

: 

## CLIP как компонент RRF

CLIP: `openai/clip-vit-base-patch32` (512 dim), 15530 мемов

| Конфиг | Hit@1 EN | Hit@5 EN | Hit@10 EN | MRR EN |
|---|:---:|:---:|:---:|:---:|
| F (без CLIP) | 29.9% | 60.0% | 65.5% | 0.429 |
| **F+CLIP [0.5,0.3,0.2]** | **30.5%** | **60.4%** | **68.7%** | **0.438** |

| Конфиг | Hit@1 RU | Hit@5 RU | Hit@10 RU | MRR RU |
|---|:---:|:---:|:---:|:---:|
| **F (без CLIP)** | **26.3%** | **57.0%** | **60.8%** | **0.397** |
| F+CLIP [0.5,0.3,0.2] | 24.0% | 54.9% | 60.6% | 0.374 |

**Вывод:** CLIP улучшает EN (Hit@10 +3.2%), но ухудшает RU (Hit@5 −2.1%) — CLIP text encoder не мультиязычный. Для RU нужен `clip-ViT-B-32-multilingual-v1`.
